# 기준 요청과 현재 변화 조건의 입력 분포 비교

기준 모델의 `high_risk` 예측 비율이 늘었다면 모델부터 탓하지 않고 입력이 달라졌는지 먼저 확인합니다. 이 노트북은 정답이 없는 운영 요청 표본에 `current-shift` 변환을 적용하고, 네 특성의 분포 변화를 Pandas로 비교합니다. 이 결과는 입력 변화의 증거이지 모델 성능 저하나 원인의 확정 근거가 아닙니다.

In [ ]:
from pathlib import Path

import pandas as pd
import yaml

ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
    and (parent / "configs/traffic/scenarios.yaml").exists()
)
OPERATIONAL_DATA = ROOT / "data/splits/physionet-2012/revisions/v2/datasets/operational.csv"
SCENARIOS = ROOT / "configs/traffic/scenarios.yaml"

baseline = pd.read_csv(OPERATIONAL_DATA)
scenario_config = yaml.safe_load(SCENARIOS.read_text(encoding="utf-8"))
transforms = scenario_config["scenarios"]["current-shift"]["transforms"]

print(f"rows={len(baseline)}, columns={len(baseline.columns)}")
print(f"target_present={'target' in baseline.columns}")
pd.DataFrame(transforms).T

운영 요청 표본에는 `target`이 없어야 합니다. 정답이 없는 자료로 새 재현율이나 정밀도를 계산하지 않기 위해서입니다. `current-shift`는 나이, 평균 심박수, 마지막 젖산값, 최소 GCS에 정해진 변환만 적용합니다.

In [ ]:
current_shift = baseline.copy()
for feature, rule in transforms.items():
    values = current_shift[feature].astype(float)
    values = values * float(rule.get("multiply", 1.0))
    values = values + float(rule.get("add", 0.0))
    if "minimum" in rule:
        values = values.clip(lower=float(rule["minimum"]))
    if "maximum" in rule:
        values = values.clip(upper=float(rule["maximum"]))
    current_shift[feature] = values

assert "target" not in baseline.columns
assert baseline["record_id"].equals(current_shift["record_id"])
assert len(baseline) == len(current_shift)
print("동일한 100개 요청에 설정된 변환만 적용했습니다.")

두 표본의 레코드와 나머지 특성을 그대로 둔 채 네 특성만 바뀌었는지 확인합니다. 비교 조건을 고정해야 예측 분포가 달라졌을 때 입력 변화와 모델 교체의 효과를 섞지 않을 수 있습니다.

In [ ]:
rows: list[dict[str, float | int | str]] = []
for feature in transforms:
    rows.append(
        {
            "feature": feature,
            "observed_rows": int(baseline[feature].notna().sum()),
            "baseline_mean": round(float(baseline[feature].mean()), 3),
            "shift_mean": round(float(current_shift[feature].mean()), 3),
            "baseline_median": round(float(baseline[feature].median()), 3),
            "shift_median": round(float(current_shift[feature].median()), 3),
        }
    )

comparison = pd.DataFrame(rows)
comparison

평균과 중앙값이 설정한 방향으로 움직이면 입력 조건이 달라졌다는 원인 후보는 강화됩니다. 다만 결측이 있는 특성은 관측 행 수가 100보다 작으므로 분모를 함께 기록해야 합니다. 또한 이 표만으로 실제 운영에서 같은 변화가 발생했다고 말할 수 없으며, 대상 환경의 시나리오와 시간 범위를 별도로 확인해야 합니다.

In [ ]:
changed_columns = [
    column
    for column in baseline.columns
    if not baseline[column].equals(current_shift[column])
]
expected_columns = list(transforms)
print(f"changed_columns={changed_columns}")
print(f"expected_columns={expected_columns}")
assert set(changed_columns) == set(expected_columns)

evidence = {
    "fact": "설정된 네 특성의 입력 분포가 기준 요청과 달라졌다",
    "cause_candidate": "입력 조건 변화: 강화",
    "not_proven": [
        "모델 성능 저하",
        "실제 운영 환경에서 같은 변화가 발생함",
        "예측 비율 증가의 단일 원인",
    ],
    "next_evidence": "같은 모델 정보와 시간 범위의 점수, 예측 분포, 요청 기록",
}
evidence